# Time-dependent toy example

Five-timestep electricity dispatch with a battery, demonstrating `pulpo.utils.time_extension`. The five steps represent a stylised day: early morning, mid-morning, midday, afternoon and evening.

Activities: `solar`, `coal`, `battery_charge`, `battery_discharge`. Demand is placed on a virtual `electricity` product formed by PULPO `choices` over `{solar, coal, battery_discharge}`. Only `coal` emits CO2 (1 kg/kWh); the only LCIA method is GWP100.

The storage spec `(battery_charge, battery_charge, K)` says: charging at *t-1* makes `K` units of stored energy available at *t* (one-step lag with round-trip efficiency `K`).

In [ ]:
import bw2data as bd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pulpo import pulpo_time
from pulpo.datasets.elec_time_database import (
    setup_elec_time_db,
    PROJECT_NAME, DB_NAME,
    SOLAR_KEY, COAL_KEY, BATTERY_CHARGE_KEY, BATTERY_DISCHARGE_KEY,
)

setup_elec_time_db()
bd.projects.set_current(PROJECT_NAME)
db = bd.Database(DB_NAME)

solar     = next(a for a in db if a.key == SOLAR_KEY)
coal      = next(a for a in db if a.key == COAL_KEY)
charge    = next(a for a in db if a.key == BATTERY_CHARGE_KEY)
discharge = next(a for a in db if a.key == BATTERY_DISCHARGE_KEY)

## Define the time-dependent problem

Solar capacity follows a bell shape peaking at noon. Demand is moderate during the day and rises in the evening when there is no sun left. Coal is always available but expensive in CO2 terms.

In [ ]:
GWP100 = str(("GWP", "100a"))

time_steps = [0, 1, 2, 3, 4]
step_labels = ["early\nmorning", "mid-\nmorning", "midday", "after-\nnoon", "evening"]

solar_cap  = {0:  10.0, 1:  80.0, 2: 120.0, 3:  60.0, 4:   0.0}
demand_kwh = {0:  50.0, 1:  60.0, 2:  60.0, 3:  160.0, 4: 100.0}

choices = {"electricity": {solar: 1e6, coal: 1e6, discharge: 1e6}}

# Per-timestep capacities. Discharge is locked at t=0 because the battery starts empty.
upper_limit = {
    t: {
        solar: solar_cap[t],
        coal: 1e6,
        charge: 1e6,
        discharge: 0.0 if t == 0 else 1e6,
    }
    for t in time_steps
}

# Battery flows must be physically non-negative. The default lower bound is
# -inf, which would let the optimizer run `charge` at a negative level at the
# last step (effectively a free electricity source, since charging is a sink).
lower_limit = {
    t: {charge: 0.0}
    for t in time_steps
}

demand = {t: {"electricity": demand_kwh[t]} for t in time_steps}

### Visualize the inputs

In [ ]:
x = np.arange(len(time_steps))
width = 0.4

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(x - width/2, [solar_cap[t]  for t in time_steps], width,
       label="solar capacity", color="#f4a826")
ax.bar(x + width/2, [demand_kwh[t] for t in time_steps], width,
       label="demand",          color="#4c72b0")
ax.set_xticks(x); ax.set_xticklabels(step_labels)
ax.set_ylabel("kWh per step")
ax.set_title("Inputs: solar availability vs. electricity demand")
ax.legend(loc="upper left")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout(); plt.show()

## Solve three scenarios

1. **No battery** -- storage disabled.
2. **Lossy battery** ($K = 0.9$) -- carry-over with round-trip losses.
3. **Ideal battery** ($K = 1.0$) -- lossless upper bound.

Note: storage couples consecutive steps only, so charging at *t-1* shows up at *t*. To carry energy from midday to evening the optimizer cascades: discharge at the afternoon to free solar for charging, then discharge in the evening.

In [ ]:
def solve_scenario(label, *, K):
    storage_spec = [(charge, charge, K)] if K is not None else None
    w = pulpo_time.PulpoOptimizerTime(PROJECT_NAME, DB_NAME, {GWP100: 1}, ".")
    w.get_lci_data()
    w.instantiate(
        choices=choices,
        demand=demand,
        upper_limit=upper_limit,
        lower_limit=lower_limit,
        time_steps=time_steps,
        storage=storage_spec,
    )
    w.solve()
    pmap = w.lci_data["process_map"]
    df = pd.DataFrame(
        {
            name: [w.instance.scaling_vector[t, pmap[act.key]].value for t in time_steps]
            for name, act in [("solar", solar), ("coal", coal),
                              ("charge", charge), ("discharge", discharge)]
        },
        index=step_labels,
    )
    df["co2"] = [w.instance.impacts[t, GWP100].value for t in time_steps]
    df.attrs["label"] = label
    df.attrs["K"] = K
    df.attrs["total_co2"] = float(df["co2"].sum())
    return w, df

_, df_nobattery = solve_scenario("no battery",            K=None)
_, df_battery   = solve_scenario("battery (K = 0.3)",      K=0.3)
_, df_ideal     = solve_scenario("ideal battery (K = 1.0)", K=1.0)

for df in (df_nobattery, df_battery, df_ideal):
    print(f"  {df.attrs['label']:30s}  total CO2 = {df.attrs['total_co2']:7.2f} kg")

### Visualize the dispatch

Each subplot has two halves:

* **Above zero -- supply meeting demand.** Stacked bars show how the demand
  at each step is met: `solar` and `coal` produced in this step, plus
  `discharge` drawn out of the battery. The black tick marks the demand level.
* **Below zero -- electricity diverted into storage.** The hatched green bar
  is `charge`, plotted downward to make it visually obvious that this is
  electricity *consumed* (a sink), not supplied. It is balanced against the
  upward bars: at each step, (solar + coal generated) = (demand met directly)
  + (charge into battery), and (discharge) = `K * charge[t-1]`.

The dashed line shows the stored energy carried *into* each step
(`K * charge[t-1]`), i.e. what is available for `discharge` that step.

In [ ]:
def stored_energy_series(df):
    K = df.attrs["K"] or 0.0
    charge_prev = [0.0] + df["charge"].iloc[:-1].tolist()
    return [K * c for c in charge_prev]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), sharey=True)
for ax, df in zip(axes, [df_nobattery, df_battery, df_ideal]):
    # --- Supply side (positive): stacks up to meet demand ---
    bottoms = np.zeros(len(time_steps))
    for col, color, label in [
        ("solar",     "#f4a826", "solar"),
        ("discharge", "#9b59b6", "discharge (out of battery)"),
        ("coal",      "#5d5d5d", "coal"),
    ]:
        vals = df[col].values
        ax.bar(x, vals, bottom=bottoms, label=label, color=color, edgecolor="white")
        bottoms = bottoms + vals

    # --- Storage side (negative): electricity diverted into the battery ---
    charge_vals = df["charge"].values
    ax.bar(x, -charge_vals, bottom=0.0, label="charge (into battery, sink)",
           color="none", edgecolor="#2ca02c", hatch="//", linewidth=1.0)

    # Stored energy carried into each step (available for discharge that step)
    if df.attrs["K"] is not None:
        ax.plot(x, stored_energy_series(df), "o--", color="#2ca02c",
                label="stored energy")

    # Demand reference line
    ax.plot(x, [demand_kwh[t] for t in time_steps], "k_", markersize=18,
            markeredgewidth=2, label="demand")

    ax.axhline(0, color="black", linewidth=0.6)
    ax.set_xticks(x); ax.set_xticklabels(step_labels)
    ax.set_title(f"{df.attrs['label']}\nCO2 = {df.attrs['total_co2']:.1f} kg")
    ax.grid(axis="y", alpha=0.3)

axes[0].set_ylabel("kWh per step\n(- charging into battery   |   + supplied to demand +)")
axes[-1].legend(loc="upper left", fontsize=8, bbox_to_anchor=(1.02, 1.0))
fig.suptitle("Dispatch comparison")
fig.tight_layout(); plt.show()